# V13 — Symbolic IR SFT (v2)

Goal: convert reasoning into structured symbolic intermediate representation (parse → transform → constraints → boxed answer).

In [ ]:
import pandas as pd
import re
import random

TRAIN_PATH = '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv'
df = pd.read_csv(TRAIN_PATH)
print(df.shape)

In [ ]:
# -----------------------------
# Step 1: category tagging
# -----------------------------

def cat(x):
    x = str(x).lower()
    if 'bit' in x: return 'bit'
    if 'cipher' in x: return 'cipher'
    if 'unit' in x: return 'unit'
    if 'gravity' in x: return 'gravity'
    if 'equation' in x: return 'equation'
    return 'other'

df['cat'] = df['prompt'].apply(cat)
df['cat'].value_counts()

In [ ]:
# -----------------------------
# Step 2: symbolic IR builder
# -----------------------------

def build_ir(prompt, answer):
    return f'''[PARSE]
{prompt[:140]}

[SYMBOL_MAP]
OP→φ VAR→α NUM→β

[STATE]
initialized=True

[CONSTRAINTS]
consistent=True valid_path=True

[EXEC]
symbolic_transform

[OUTPUT]
\boxed{{{answer}}}'''

In [ ]:
# -----------------------------
# Step 3: small synthetic IR (safe categories only)
# -----------------------------

cats = ['bit','cipher','unit','gravity']
synthetic = []

for c in cats:
    sub = df[df['cat']==c].sample(min(200, len(df[df['cat']==c])), random_state=42)
    for _, r in sub.iterrows():
        synthetic.append({
            'prompt': r['prompt'],
            'answer': r['answer'],
            'text': build_ir(r['prompt'], r['answer'])
        })

print('synthetic:', len(synthetic))

In [ ]:
# -----------------------------
# Step 4: merge dataset
# -----------------------------

rows = df.to_dict('records') + synthetic
print('total:', len(rows))

In [ ]:
# -----------------------------
# Step 5: training config
# -----------------------------

CONFIG = {
    'warmstart': '/kaggle/input/notebooks/jatalepawan/v2-end-to-end-finetuning-lb082-bit-reweight/submission.zip',
    'lr': 7e-7,
    'steps': 56,
    'reset': False
}

print(CONFIG)